# Notebook 04 — Exposure Timeline Generation
**GeoAI for Climate, Environment and Health** | 25 June 2026

---

## Learning objectives
- Understand why environmental exposure in longitudinal studies is a *time-varying location* problem
- Detect participant migration between study visits
- Implement a rule that assigns every day of pregnancy to the nearest-in-time visit location
- Produce `participant_exposure_days.csv` — the central artifact used by later notebooks

**Time:** ~25 minutes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys, pathlib, os

def _find_workshop_dir():
    here = pathlib.Path().resolve()
    for candidate in (here, here.parent, here.parent.parent):
        if (candidate / 'data').exists():
            return candidate
    return here

WORKSHOP_DIR = _find_workshop_dir()  # works whether run flat or from a notebooks/ subfolder
DATA_DIR = WORKSHOP_DIR / 'data'
sys.path.insert(0, str(WORKSHOP_DIR))
from shared_utils import assign_visit_location_to_days, assign_trimester

# Always read PID as string.
participants = pd.read_csv(DATA_DIR / 'participants.csv', dtype={'PID': str})
visits       = pd.read_csv(DATA_DIR / 'visits.csv',       dtype={'PID': str})
villages     = pd.read_csv(DATA_DIR / 'villages.csv')

print(f'Participants: {len(participants)}')
print(f'Visits:       {len(visits)}  ({visits["visit_type"].value_counts().to_dict()})')
print(f'Villages:     {len(villages)}')

---
## 📖 1. The time-varying location problem

In a pregnancy cohort, we want to know what air quality, temperature, and vegetation a woman was exposed to during her pregnancy — from conception to delivery.

The challenge: **we do not know exactly where she was every single day.** We only have three location snapshots — one per study visit. And for some participants, those locations are *different*: she may have been in Riverside at visit 1, moved to Ironstone by visit 2, and delivered at Kopje.

**Our approach:** for every day of pregnancy, assign the location of whichever visit was *nearest in time* to that day. We use midpoint breakpoints between consecutive visit dates.

```
Conception     Visit 1       Visit 2      Delivery
    |------------|------------|------------|---->
         ↑           ↑              ↑
    village A    village A      village B

Breakpoints:      ×                ×
Windows:  [──A──][─────A──────][────B────]
```

This produces a table with one row per (participant, day) — ~17,000 rows for 60 participants over ~280 days each.

---
## ✅ Worked example 1 — Look at one migrant participant's three visits

In [ ]:
# Find participants who migrated (reported different villages across visits).
migrants = participants[participants['migrated'] == True]['PID'].tolist()
print(f'Migrant participants: {len(migrants)} of {len(participants)}')

# Pick the first migrant to inspect.
example_pid = migrants[0]
print(f'\nVisits for participant {example_pid}:')
ex_visits = visits[visits['PID'] == example_pid].sort_values('visit_date')
print(ex_visits[['visit_type', 'visit_date', 'village', 'latitude', 'longitude']].to_string(index=False))

---
## 🧪 Exercise 1 — Find all migrants without using the `migrated` column  *(3 points)*

Without looking at `participants['migrated']`, identify all `PID`s whose three visits report **more than one distinct village**.

Store the result as a Python set (or list) called `detected_migrants`. Then compare it to the `migrated` column as a sanity check.

In [ ]:
# Hint: group participants' visits by PID, then count the number of
# DISTINCT villages reported per PID with .nunique(). Any PID with more
# than 1 distinct village has moved. Store the matching PIDs as a set
# in detected_migrants.

village_counts = visits.groupby('PID')['village'].nunique()
detected_migrants = set(village_counts[village_counts > 1].index.tolist())

true_migrants = set(participants[participants['migrated'] == True]['PID'].tolist())
print(f'Detected by code: {len(detected_migrants)} migrants')
print(f'Flagged in data:  {len(true_migrants)} migrants')
print(f'Match: {detected_migrants == true_migrants}')


---
### ✅ Knowledge Check — Detecting residential mobility

*No coding here — just check your understanding before moving on.*

**Q1. Why does it matter, for an environmental exposure study, whether a pregnant participant moved between villages during her pregnancy?**

- **A.** It doesn't matter — environmental exposure is the same everywhere
- **B.** If she moved, her true exposure changed over time, so assigning her a single fixed location for the whole pregnancy would misclassify part of her exposure
- **C.** Movement only matters for migration researchers, not exposure scientists
- **D.** Moving participants should simply be excluded from every analysis

<details>
<summary>Show answer</summary>

**Answer: B** — A participant who lived in a hot, polluted area for trimester 1 and a cooler area for trimester 3 had two different exposure profiles — using one static location for the whole pregnancy would be exposure misclassification.

</details>

**Q2. Why is it useful to be able to *detect* migration from visit records, rather than only trusting a pre-existing 'migrated' flag in the data?**

- **A.** It isn't useful — flags in datasets are always complete and correct
- **B.** Real-world datasets can have missing, outdated, or inconsistent flags, so being able to derive the same information independently is a way to check data quality
- **C.** Detecting migration this way is purely an academic exercise with no data-quality use
- **D.** Visit records can never disagree with a migration flag

<details>
<summary>Show answer</summary>

**Answer: B** — Cross-checking a derived result against an existing flag (as this exercise does) is a basic data-quality habit — it catches cases where the recorded flag might be wrong or stale.

</details>



In [ ]:
# Tests hidden — run your solution above first
pass

---
## ✅ Worked example 2 — Build the pregnancy day range per participant

In [ ]:
participants['conception_date']       = pd.to_datetime(participants['conception_date'])
participants['actual_delivery_date']  = pd.to_datetime(participants['actual_delivery_date'])
participants['expected_delivery_date']= pd.to_datetime(participants['expected_delivery_date'])

rows = []
for _, p in participants.iterrows():
    start = p['conception_date']
    end   = p['actual_delivery_date'] if pd.notna(p['actual_delivery_date']) else p['expected_delivery_date']
    for gday, dt in enumerate(pd.date_range(start, end)):
        rows.append({'PID': p['PID'], 'date': dt.date(), 'gestational_day': gday})

preg_days = pd.DataFrame(rows)
print(f'Pregnancy-day table: {len(preg_days):,} rows')
print(f'Gestational day range: {preg_days["gestational_day"].min()} – {preg_days["gestational_day"].max()}')
preg_days.head()

> **Scale note:** ~60 participants × ~280–310 days ≈ 17,000 rows — comfortably handled by pandas. In a real study with thousands of participants and multi-year follow-up, this same per-day, per-location logic would generate tens of millions of rows — which is why checkpointing and incremental processing matter in production pipelines.

---
## ✅ Worked example 3 — Assign each day to the nearest-in-time visit location

In [ ]:
# assign_visit_location_to_days() is implemented in shared_utils.py.
# It:
#   1. Sorts each participant's 3 visits by date.
#   2. Computes 2 midpoint breakpoints between consecutive visits.
#   3. Assigns village/lat/lon based on which window each day falls in.
#   4. Records which visit was the source (source_visit_type).
# Breakpoint rule: a day exactly on a breakpoint goes to the LATER visit.

preg_days = assign_visit_location_to_days(preg_days, visits)
preg_days['trimester'] = assign_trimester(preg_days['gestational_day'])

print(f'Columns after assignment: {list(preg_days.columns)}')
print()
print('Source visit breakdown (days assigned to each visit window):')
print(preg_days['source_visit_type'].value_counts())
preg_days.head()

---
## 🧪 Exercise 2 — Step plot for one migrant participant  *(3 points)*

For the same migrant participant you inspected in Worked example 1 (`example_pid`):
1. Filter `preg_days` to just that participant.
2. Plot `gestational_day` on the x-axis and `village` on the y-axis as a **step plot** (use `drawstyle='steps-post'` in matplotlib).
3. The plot should clearly show a step (a sudden jump between villages) somewhere during the pregnancy.

Store the participant's filtered data in `example_days`.

In [ ]:
example_days = preg_days[preg_days['PID'] == example_pid].sort_values('gestational_day')

# Hint: plt.step(example_days['gestational_day'], example_days['village'], where='post')
# (or plt.plot(..., drawstyle='steps-post')) then plt.show().

plt.figure(figsize=(10, 3))
plt.plot(example_days['gestational_day'], example_days['village'], drawstyle='steps-post',
         color='#2a9d8f', linewidth=2)
plt.xlabel('Gestational day')
plt.ylabel('Assigned village')
plt.title(f'Location assignment over pregnancy — participant {example_pid}')
plt.tight_layout()
plt.show()

print(f'Villages in pregnancy: {example_days["village"].unique()}')


---
### ✅ Knowledge Check — Visualising exposure over time

*No coding here — just check your understanding before moving on.*

**Q1. In the step plot, what does a sudden 'step' (jump) in the village line during the pregnancy represent?**

- **A.** A data entry error that should be deleted
- **B.** The point in the pregnancy when the participant physically relocated to a different village
- **C.** A change in the participant's age
- **D.** The day she gave birth

<details>
<summary>Show answer</summary>

**Answer: B** — Each step corresponds to a change in reported village — i.e. a residential move — which is exactly the kind of event that breaks a 'one location for the whole pregnancy' assumption.

</details>



In [ ]:
# Tests hidden — run your solution above first
pass

---
## ✅ Save the output

In [ ]:
output_dir = DATA_DIR if os.access(DATA_DIR, os.W_OK) else pathlib.Path.home()
if output_dir != DATA_DIR:
    print(f'Note: {DATA_DIR} is read-only here (you may be running from the shared course '
          f'copy rather than your own). Saving to your home directory instead.')

output_path = output_dir / 'participant_exposure_days_notebook.csv'
preg_days.to_csv(output_path, index=False)
print(f'Saved {len(preg_days):,} rows to {output_path}')
print('Columns:', list(preg_days.columns))


---
## 🏁 Recap

- In a longitudinal cohort, a participant's location is not fixed — it must be inferred from sparse visit records.
- The nearest-in-time midpoint rule is a principled, simple way to assign daily locations.
- Migrants (≈30% of participants) will have a step in their location trajectory — this produces a real difference in computed exposure between trimesters.
- We now have a table of ~17,000 rows linking every participant × day to a geographic location.

**Next:** Notebook 05 explains where environmental data comes from and loads the pre-generated exposure grid.